# 02 From Design to Value

How do a property's spaces, buildings and shared systems produce income, costs and value?

Notebook 01 built a DCF from financial assumptions. Here, we connect those assumptions to a building project. We will inspect its entities and relationships, calculate how each entity contributes to aggregated cash flows, and explain how changes in input parameters change overall valuation.

## 1. Meet and load the project

**Building A** contains hotel, residential, retail and parking spaces. **Building B** contains residential, retail and parking spaces. They share a basement plinth and plant; each building has cores.

Treat the design as a proposed **completed property**. Which parts earn income, and which enable other parts to operate? Development timing and funding follow in notebook 03.

In the 3D viewer, switch from **Whole context** to **Property**, then **Floors**. Rotate, zoom and select a floor to inspect its recorded properties. Geometry and graph selections are independent.

Locate the two buildings and basement before reading the graph. Use the object name and ID to follow the same component between views.


In [1]:
from helpers.plot import show_geometry

from IPython.display import IFrame, display

display(
    show_geometry("context")
)


Geometry describes form; a graph describes objects and relationships. The following diagram connects these two descriptions:

![How physical property components become graph objects and relationships.](resources/devModelGraph.jpg)

*This concept illustration is not the exact graph of our two-building project. Inspect that graph below.*

A **Model** holds definitions, objects, relationships and provenance. An **Entity** records an object; an **Assembly** also declares a grouping. A **Value** records a characteristic such as area or use.

![Current records and their relationship to numerical calculations.](resources/mixed-use/object-model.svg)

Load the pinned Model, then calculate flows separately using object IDs. Source measurements and financial assumptions remain separate.


In [2]:
from pathlib import Path
from uuid import UUID
from datetime import date
import copy
import hashlib
import json
import math

import polars as pl

pl.Config.set_fmt_float("full")
pl.Config.set_thousands_separator(",")
from rangekeeper.io import json as codec
from rangekeeper.model import Model
from rangekeeper.model.characteristics import value
from rangekeeper.model.content import decode
from rangekeeper.model.definitions import find_classifications
from rangekeeper.model.system import View, Hierarchy, Reduction, reducers
from rangekeeper.model.system.selection import select_value
from helpers.plot import show_graph, show_sunburst

package = Path("data/mixed_use_property")
manifest = json.loads((package / "manifest.json").read_text())
model_bytes = (package / "model.json").read_bytes()
assert (
    hashlib.sha256(model_bytes).hexdigest() == manifest["files"]["model.json"]["sha256"]
)
model = codec.read(package / "model.json", kind=Model)
source_before = codec.dumps(model)
output = Path("design-output/notebook-02")
output.mkdir(parents=True, exist_ok=True)

print(f"Model revision: {model.id}")
print(f"Source captured: {manifest['captured_on']}")


Model revision: 37565d7c-f2fb-5ddd-9c4d-e5faacd5c2e0
Source captured: 2026-10-12


The source contains **50 domain objects**. The geometry archive also contains surrounding context, which does not enter the property calculation.

Drag nodes, zoom and read edge labels in the graph. Find `buildingA`, `buildingB`, `plinth` and `utilities`. Arrows identify relationships; they do not prescribe a financial solve direction.


In [3]:
whole_view = View(model)
show_graph(view=whole_view, path=output / "property.html")


In [4]:
def classification_id(code):
    matches = find_classifications(model.definitions, code=code)
    if len(matches) != 1:
        raise ValueError(f"Expected one classification for {code!r}")
    return matches[0].id


classifications = {
    code: classification_id(code)
    for code in ("property", "building", "space", "floor", "utilities")
}
pl.DataFrame(
    [
        {
            "Object class": code,
            "Count": sum(e.classification == uid for e in whole_view.entities),
        }
        for code, uid in classifications.items()
    ]
)


Object class,Count
str,i64
"""property""",1
"""building""",3
"""space""",9
"""floor""",33
"""utilities""",4


## 2. Read the relationships

The legacy assembly diagram shows how one object can belong to a spatial arrangement and depend on a service system.

![Overlapping spatial and HVAC assemblies.](resources/devModelAssembly.jpg)

Ask three questions: **what contains it spatially, what serves it, and which group includes it?** Our source answers these through `spatiallyContains`, `services` and `contains` relationships. A service connection does not specify cost allocation.

Select the spatial edges. `Hierarchy.from_relationships()` validates their tree structure; the display helper only draws it. The full project can be a network while a selected relationship view forms a tree. Choose the relationship to match the question.


In [5]:
spatial_id = classification_id("spatiallyContains")
spatial_view = View(
    model,
    relationships=[
        r.id for r in whole_view.relationships if r.classification == spatial_id
    ],
)
hierarchy = Hierarchy.from_relationships(spatial_view)
show_graph(view=spatial_view, path=output / "spatial.html", hierarchical=True)


Follow floor → use space → building → property. Zoom to read floor names in this 49-edge tree.

Then inspect the service chain: shared plant → building cores → served spaces. The plant also serves basement parking.


In [6]:
service_id = classification_id("services")
services = View(
    model,
    relationships=[
        r.id for r in whole_view.relationships if r.classification == service_id
    ],
)
show_graph(view=services, path=output / "services.html")


In [7]:
contains_id = classification_id("contains")
pl.DataFrame(
    [
        {"Group": model.entity(r.source).name, "Member": model.entity(r.target).name}
        for r in whole_view.relationships
        if r.classification == contains_id
    ]
)


Group,Member
str,str
"""utilities""","""buildingBcores"""
"""utilities""","""plinthplant"""
"""utilities""","""buildingAcores"""


The utilities group references three physical components: two core systems and shared plant. These are the same objects found under their spatial parents, not additional cost contributors.

## 3. From one floor to total area

Inspect Building A hotel Floor 1. GFA measures gross floor area; perimeter, floor-to-floor height and finished floor elevation describe other aspects of the same object. Units are explicit. GFA is not automatically lettable area.

Floor-to-floor height will later help estimate facade costs. Finished floor elevation locates the floor; it does not itself enter the income formula.


In [8]:
assumptions = json.loads((package / "assumptions.json").read_text())
floor = model.entity(UUID(assumptions["worked_floor_id"]))


# A recorded measurement carries units. Fail if it is absent.
def quantity(entity, key, units):
    measurement = value(entity.characteristics, key)
    if measurement is None or measurement.quantity is None:
        raise ValueError(f"{entity.name}: missing {key}")
    result = measurement.quantity
    if result.units != units:
        raise ValueError(f"{entity.name}: expected {units}, got {result.units}")
    return result.magnitude


pl.DataFrame(
    [
        {
            "Floor": floor.name,
            "Quantity": key,
            "Magnitude": quantity(floor, key, units),
            "Units": units,
        }
        for key, units in (
            ("gfa", "meter ** 2"),
            ("perimeter", "meter"),
            ("ftf", "meter"),
            ("ffl", "meter"),
        )
    ]
)


Floor,Quantity,Magnitude,Units
str,str,f64,str
"""buildingAhotelFloor1""","""gfa""","1,697.4515719337053","""meter ** 2"""
"""buildingAhotelFloor1""","""perimeter""",171.07782517573892,"""meter"""
"""buildingAhotelFloor1""","""ftf""",3.1000000000000005,"""meter"""
"""buildingAhotelFloor1""","""ffl""",5.3,"""meter"""


Select by identity within the building/use context. Names can repeat: Building B contains two distinct residential Floor 0 objects.

For area aggregation, select **floor objects only**. Parent totals are derived, not extra contributors. `require_complete=True` checks that selected contributors have the required quantity.


In [9]:
floors = [
    e for e in whole_view.entities if e.classification == classifications["floor"]
]
gfa = Reduction(
    select=select_value("gfa"),
    reducer=reducers.sum,
    units="meter ** 2",
    contributors=lambda entity: entity.classification == classifications["floor"],
    require_complete=True,
).execute(hierarchy)
assert gfa.coverage(hierarchy.root).complete
assert len(floors) == 33

branches = {
    "Building A": UUID("5ebb9736-5f18-4103-a5a4-040cd2e9b019"),
    "Building B": UUID("a1e00481-e51e-47c5-b629-8b214c0900bd"),
    "Shared plinth": UUID("0a0a97ef-431d-471d-bc33-ace7e7df5fcb"),
}
pl.DataFrame(
    [
        {"Component": label, "GFA (m²)": gfa.value(uid).magnitude}
        for label, uid in {**branches, "Property": hierarchy.root}.items()
    ]
).with_columns(pl.col("GFA (m²)").round(2))


Component,GFA (m²)
str,f64
"""Building A""","16,575.63"
"""Building B""","13,436.88"
"""Shared plinth""","12,773.74"
"""Property""","42,786.26"


The property has **42,786.26 m² across 33 floor objects**. Objects can share elevations, so this is not a count of storeys.

The sunburst shows the spatial hierarchy and each part's area. Click a building or use to drill down, click the centre to return, and hover for values. We will reuse this view for income and costs. Presentation settings live in `helpers/plot.py`.

Before moving on, find the shared plinth in the chart. Predict whether its large share of area will also give it a large share of income.


In [10]:
floor_areas = {e.id: quantity(e, "gfa", "meter ** 2") for e in floors}
show_sunburst(
    hierarchy=hierarchy,
    amounts=floor_areas,
    title="Gross floor area · all 33 floors",
    units="m²",
    path=output / "gfa-sunburst.html",
)


## 4. From area to an income flow

For Building A hotel Floor 1, income-producing area = GFA × efficiency. Potential gross income (PGI) = income-producing area × annual rent per m².

Efficiency and rent are teaching assumptions from the legacy example, not current market estimates. A floor's use comes from its spatial parent.


In [11]:
def source_use(entity):
    recorded = value(entity.characteristics, "use")
    return None if recorded is None else decode(recorded.content)


def floor_use(entity):
    parent = model.entity(hierarchy.parent(entity.id))
    use = source_use(parent)
    if use not in assumptions["use_assumptions"]:
        raise ValueError(f"{entity.name}: no supported use on its spatial parent")
    return use


use = floor_use(floor)
hotel = assumptions["use_assumptions"][use]
area = quantity(floor, "gfa", "meter ** 2")
efficient_area = area * hotel["efficiency"]
initial_income = efficient_area * hotel["initial_rent_AUD_per_m2_per_year"]

pl.DataFrame(
    {
        "GFA (m²)": [area],
        "Efficiency": [hotel["efficiency"]],
        "Income area (m²)": [efficient_area],
        "Rent (AUD/m²/year)": [hotel["initial_rent_AUD_per_m2_per_year"]],
        "Initial PGI (AUD/year)": [initial_income],
    }
).with_columns(pl.col(pl.Float64).round(2))


GFA (m²),Efficiency,Income area (m²),Rent (AUD/m²/year),Initial PGI (AUD/year)
f64,f64,f64,i64,f64
"1,697.45",0.8,"1,357.96",750,"1,018,470.94"


Initial PGI is **AUD 1,018,470.94 per year**. Optional exercise: predict and calculate the change if efficiency falls from 80% to 75%, while GFA and rent stay fixed.


Now use notebook 01's flow tools. Hold for **2001–2010** and forecast **2011** for the sale calculation. Receipts and costs occur at year end; the first year uses the initial amount before growth. Vacancy is a signed deduction, so EGI = PGI + vacancy.


In [12]:
from rangekeeper.model.duration import Span, Frequency, PeriodTiming
from rangekeeper.model.flux import Flow, Stream, MissingValueHandling
from rangekeeper.model.measure import Quantity
from rangekeeper.calculations import projection, financial
from rangekeeper.calculations.series import ResamplingMethod

holding_span = Span(
    name="Holding",
    start_inclusive=date.fromisoformat(assumptions["holding_span"]["start_inclusive"]),
    end_exclusive=date.fromisoformat(assumptions["holding_span"]["end_exclusive"]),
)
forecast_span = Span(
    name="Forecast including sale basis",
    start_inclusive=holding_span.start_inclusive,
    end_exclusive=date.fromisoformat(assumptions["forecast_span"]["end_exclusive"]),
)
annual_periods = forecast_span.periods(frequency=Frequency.YEAR)

pgi = projection.extrapolate(
    initial=Quantity(magnitude=initial_income, units="AUD"),
    periods=annual_periods,
    method=projection.ProjectionMethod.COMPOUND,
    rate=hotel["annual_income_growth"],
)
vacancy = pgi.scale(factor=hotel["vacancy"]).negate()
egi = Stream(flows={"PGI": pgi, "Vacancy": vacancy}).sum()
Stream(flows={"PGI": pgi, "Vacancy": vacancy, "EGI": egi}).display()


Period / date,PGI [AUD],Vacancy [AUD],EGI [AUD]
2001,"1,018,470.94","-25,461.77","993,009.17"
2002,"1,054,117.43","-26,352.94","1,027,764.49"
2003,"1,091,011.54","-27,275.29","1,063,736.25"
2004,"1,129,196.94","-28,229.92","1,100,967.02"
2005,"1,168,718.83","-29,217.97","1,139,500.86"
2006,"1,209,623.99","-30,240.60","1,179,383.39"
2007,"1,251,960.83","-31,299.02","1,220,661.81"
2008,"1,295,779.46","-32,394.49","1,263,384.97"
2009,"1,341,131.74","-33,528.29","1,307,603.45"
2010,"1,388,071.35","-34,701.78","1,353,369.57"


The floor's 2001 effective gross income (EGI) is **AUD 993,009.17**. Check that deduction and compare the two flows over time.


In [13]:
from helpers.plot import show_flows

assert math.isclose(egi.movements[0].number, initial_income * (1 - hotel["vacancy"]))
show_flows(
    flows={"Potential income": pgi, "Effective income": egi},
    title="Building A hotel Floor 1 · annual income",
    timing=PeriodTiming.LAST,
    path=output / "annual-income.html",
)


Hover for exact dates and amounts; click a legend label to hide or show a flow. Both flows grow at 3.5% per year; vacancy remains a fixed proportion of PGI.

## 5. Apply the rule across the property

Show the assumptions for every use, including parking's explicit zero rent. Apply the worked rule to all floors, retaining each Flow against its object's ID.


In [14]:
use_inputs = assumptions["use_assumptions"]
pl.DataFrame(
    [
        {
            "Use": use,
            "Efficiency": a["efficiency"],
            "Rent (AUD/m²/year)": a["initial_rent_AUD_per_m2_per_year"],
            "Growth (%/year)": 100 * a["annual_income_growth"],
            "Vacancy (%)": 100 * a["vacancy"],
        }
        for use, a in use_inputs.items()
    ]
)


Use,Efficiency,Rent (AUD/m²/year),Growth (%/year),Vacancy (%)
str,f64,i64,f64,f64
"""hotel""",0.8,750,3.5000000000000004,2.5
"""parking""",0.9,0,0,0
"""residential""",0.75,600,5.5,1.5
"""retail""",0.675,"1,000",7.5,7.5


The function repeats the worked calculation. The mapping keeps all floor identities, so we can later select a building or use without rebuilding the source Model. A missing or unsupported use raises an error; it is not inferred from the object's name.


In [15]:
def floor_income(entity, inputs):
    use = floor_use(entity)
    a = inputs[use]
    initial = (
        quantity(entity, "gfa", "meter ** 2")
        * a["efficiency"]
        * a["initial_rent_AUD_per_m2_per_year"]
    )
    potential = projection.extrapolate(
        initial=Quantity(magnitude=initial, units="AUD"),
        periods=annual_periods,
        method=projection.ProjectionMethod.COMPOUND,
        rate=a["annual_income_growth"],
    )
    deduction = potential.scale(factor=a["vacancy"]).negate()
    return Stream(flows={"PGI": potential, "Vacancy": deduction}).sum()


income_by_floor = {e.id: floor_income(e, use_inputs) for e in floors}
assert len(income_by_floor) == len(floors)
property_egi = Stream(
    flows={str(uid): flow for uid, flow in income_by_floor.items()}
).sum()


In [16]:
initial_egi = {uid: flow.movements[0].number for uid, flow in income_by_floor.items()}
show_sunburst(
    hierarchy=hierarchy,
    amounts=initial_egi,
    title="Effective gross income · 2001",
    units="AUD/year",
    path=output / "income-sunburst.html",
)


Compare income with the GFA sunburst. The plinth occupies considerable area but earns no rent. Other contributions depend on efficiency, rent and vacancy. Zero-value branches have no visible segment; their source objects still exist.

## 6. Include operating and capital costs

**Annual operating expenditure (OPEX).** Floor costs use GFA. Facade costs use the inherited proxy **perimeter × floor-to-floor height**; the source does not establish that every boundary is external facade. Cost rates are negative.


In [17]:
pl.DataFrame(
    [
        {
            "Use": use,
            "Floor OPEX (AUD/m²/year)": a["floor_opex_AUD_per_m2_per_year"],
            "Facade OPEX (AUD/m²/year)": a["facade_opex_AUD_per_m2_per_year"],
            "Floor CAPEX (AUD/m²/payment)": a["floor_capex_AUD_per_m2_per_payment"],
        }
        for use, a in use_inputs.items()
    ]
)


Use,Floor OPEX (AUD/m²/year),Facade OPEX (AUD/m²/year),Floor CAPEX (AUD/m²/payment)
str,i64,i64,i64
"""hotel""",-175,-50,-100
"""parking""",-65,-25,-50
"""residential""",-125,-50,-75
"""retail""",-225,-100,-150


Calculate the same hotel floor's costs, then apply the rule across the property with annual growth of 3.5%.


In [18]:
facade_area = quantity(floor, "perimeter", "meter") * quantity(floor, "ftf", "meter")
initial_floor_opex = area * hotel["floor_opex_AUD_per_m2_per_year"]
initial_facade_opex = facade_area * hotel["facade_opex_AUD_per_m2_per_year"]
initial_opex = initial_floor_opex + initial_facade_opex
pl.DataFrame(
    {
        "Facade proxy (m²)": [facade_area],
        "Floor OPEX (AUD/year)": [initial_floor_opex],
        "Facade OPEX (AUD/year)": [initial_facade_opex],
        "Total OPEX (AUD/year)": [initial_opex],
    }
).with_columns(pl.col(pl.Float64).round(2))


Facade proxy (m²),Floor OPEX (AUD/year),Facade OPEX (AUD/year),Total OPEX (AUD/year)
f64,f64,f64,f64
530.34,"-297,054.03","-26,517.06","-323,571.09"


In [19]:
def floor_opex(entity):
    a = use_inputs[floor_use(entity)]
    gfa = quantity(entity, "gfa", "meter ** 2")
    facade = quantity(entity, "perimeter", "meter") * quantity(entity, "ftf", "meter")
    initial = (
        gfa * a["floor_opex_AUD_per_m2_per_year"]
        + facade * a["facade_opex_AUD_per_m2_per_year"]
    )
    return projection.extrapolate(
        initial=Quantity(magnitude=initial, units="AUD"),
        periods=annual_periods,
        method=projection.ProjectionMethod.COMPOUND,
        rate=assumptions["annual_opex_growth"],
    )


opex_by_floor = {e.id: floor_opex(e) for e in floors}
show_sunburst(
    hierarchy=hierarchy,
    amounts={uid: flow.movements[0].number for uid, flow in opex_by_floor.items()},
    title="Operating expenditure · 2001",
    units="AUD/year",
    expense_magnitudes=True,
    path=output / "opex-sunburst.html",
)


The chart shows **expense magnitudes**; flows retain negative costs. Parking now contributes an expense.

**Scheduled capital expenditure (CAPEX).** These payments maintain the completed asset; they are not a development budget. Payments fall on **31 December 2005 and 2010**. The second is 3.5% larger than the first: growth applies per payment interval, not annually.

Create dated events and resample them into annual periods. Zero filling is explicit because the schedule specifies no payments in other years.


In [20]:
capital_payments = assumptions["capital_payments"]


def capital_flow(initial):
    events = Flow.from_events(
        dates=[date.fromisoformat(p["date"]) for p in capital_payments],
        magnitudes=[
            initial
            * (1 + assumptions["capex_growth_per_payment_interval"])
            ** p["growth_exponent"]
            for p in capital_payments
        ],
        units="AUD",
    )
    return (
        Stream(flows={"CAPEX": events})
        .resample(
            periods=annual_periods,
            method=ResamplingMethod.SUM,
            missing=MissingValueHandling.ZERO,
        )
        .sum()
    )


floor_capex = capital_flow(area * hotel["floor_capex_AUD_per_m2_per_payment"])
floor_capex.display(name="Hotel Floor 1 CAPEX")


Period / date,Hotel Floor 1 CAPEX [AUD]
2001,0.00
2002,0.00
2003,0.00
2004,0.00
2005,"-169,745.16"
2006,0.00
2007,0.00
2008,0.00
2009,0.00
2010,"-175,686.24"


**Shared services.** Inspect the cores and plant below. Their first capital payment is volume × rate: AUD −100/m³ for cores and AUD −1,000/m³ for plant. Include these components even though they produce no rent.


In [21]:
display(
    show_geometry("utilities")
)


In [22]:
utility_ids = (
    UUID("7eec5208-fc0e-4dfd-bd0b-b1d226b2e97e"),
    UUID("cb5a02f7-b1b9-4a32-95b8-bbb70a946230"),
    UUID("938f4cf1-d307-4160-9538-d4bfa9583ad8"),
)
utilities = [model.entity(uid) for uid in utility_ids]
utility_rates = assumptions["utility_capex_AUD_per_m3_per_payment"]
pl.DataFrame(
    [
        {
            "Component": e.name,
            "Volume (m³)": quantity(e, "volume", "meter ** 3"),
            "Rate (AUD/m³/payment)": utility_rates[source_use(e)],
            "First CAPEX (AUD)": quantity(e, "volume", "meter ** 3")
            * utility_rates[source_use(e)],
        }
        for e in utilities
    ]
).with_columns(pl.col(pl.Float64).round(2))


Component,Volume (m³),Rate (AUD/m³/payment),First CAPEX (AUD)
str,f64,i64,f64
"""buildingAcores""","11,086.18",-100,"-1,108,618.08"
"""buildingBcores""","8,656.63",-100,"-865,663.39"
"""plinthplant""",643.11,"-1,000","-643,105.79"


In [23]:
capex_by_component = {
    e.id: capital_flow(
        quantity(e, "gfa", "meter ** 2")
        * use_inputs[floor_use(e)]["floor_capex_AUD_per_m2_per_payment"]
    )
    for e in floors
}
capex_by_component.update(
    {
        e.id: capital_flow(
            quantity(e, "volume", "meter ** 3") * utility_rates[source_use(e)]
        )
        for e in utilities
    }
)
assert len(capex_by_component) == 36
first_capital_year = date.fromisoformat(capital_payments[0]["date"]).year
first_capex = {
    uid: next(
        m.number
        for m in flow.movements
        if m.period.start_inclusive.year == first_capital_year
    )
    for uid, flow in capex_by_component.items()
}
show_sunburst(
    hierarchy=hierarchy,
    amounts=first_capex,
    title="Capital expenditure · first payment, 2005",
    units="AUD",
    expense_magnitudes=True,
    path=output / "capex-sunburst.html",
)


The CAPEX chart includes **33 floors and three physical utilities**, each once. The cost population is wider than the income-producing population. Selecting only objects with rent would omit infrastructure needed to operate the property.

## 7. Assemble and value the property

Combine signed flows: **net operating income (NOI) = EGI + OPEX**, and **net after-capital cash flow (NACF) = NOI + CAPEX**. Display the property pro forma before calculating value.


In [24]:
property_opex = Stream(
    flows={str(uid): flow for uid, flow in opex_by_floor.items()}
).sum()
property_capex = Stream(
    flows={str(uid): flow for uid, flow in capex_by_component.items()}
).sum()
property_noi = Stream(flows={"EGI": property_egi, "OPEX": property_opex}).sum()
property_nacf = Stream(flows={"NOI": property_noi, "CAPEX": property_capex}).sum()
pro_forma = Stream(
    flows={
        "EGI": property_egi,
        "OPEX": property_opex,
        "NOI": property_noi,
        "CAPEX": property_capex,
        "NACF": property_nacf,
    }
)
pro_forma.display()


Period / date,EGI [AUD],OPEX [AUD],NOI [AUD],CAPEX [AUD],NACF [AUD]
2001,"14,402,849.55","-6,268,157.66","8,134,691.89",0.00,"8,134,691.89"
2002,"15,086,989.58","-6,487,543.18","8,599,446.40",0.00,"8,599,446.40"
2003,"15,806,139.47","-6,714,607.19","9,091,532.28",0.00,"9,091,532.28"
2004,"16,562,220.35","-6,949,618.44","9,612,601.91",0.00,"9,612,601.91"
2005,"17,357,265.38","-7,192,855.09","10,164,410.29","-5,919,092.61","4,245,317.69"
2006,"18,193,426.66","-7,444,605.02","10,748,821.65",0.00,"10,748,821.65"
2007,"19,072,982.53","-7,705,166.19","11,367,816.34",0.00,"11,367,816.34"
2008,"19,998,345.35","-7,974,847.01","12,023,498.34",0.00,"12,023,498.34"
2009,"20,972,069.80","-8,253,966.66","12,718,103.14",0.00,"12,718,103.14"
2010,"21,996,861.68","-8,542,855.49","13,454,006.19","-6,126,260.85","7,327,745.34"


NACF falls below NOI in the two capital-payment years. Reconcile Building A, Building B and the shared plinth to the property. Each building includes its core costs; the plinth includes parking and plant. Shared costs remain in that separate branch.


In [25]:
def members_below(uid):
    members = {uid}
    for child in hierarchy.children(uid):
        members.update(members_below(child))
    return members


contributors_by_branch = {label: members_below(uid) for label, uid in branches.items()}


def branch_cash_flows(income):
    return {
        label: Stream(
            flows={
                f"{metric}/{uid}": flow
                for metric, flows in (
                    ("EGI", income),
                    ("OPEX", opex_by_floor),
                    ("CAPEX", capex_by_component),
                )
                for uid, flow in flows.items()
                if uid in ids
            }
        ).sum()
        for label, ids in contributors_by_branch.items()
    }


nacf_by_branch = branch_cash_flows(income_by_floor)
reconciled = Stream(flows=nacf_by_branch).sum()
assert all(
    math.isclose(a.number, b.number, abs_tol=1e-7)
    for a, b in zip(reconciled.movements, property_nacf.movements)
)
pl.DataFrame(
    [
        {"Component": label, "2001 NACF (AUD)": flow.movements[0].number}
        for label, flow in {**nacf_by_branch, "Property": property_nacf}.items()
    ]
).with_columns(pl.col("2001 NACF (AUD)").round(2))


Component,2001 NACF (AUD)
str,f64
"""Building A""","5,526,965.2"
"""Building B""","3,490,559.29"
"""Shared plinth""","-882,832.61"
"""Property""","8,134,691.89"


The plinth has negative NACF, but its services support the rest of the property. These branch figures show contributions before any shared-cost allocation.

Apply notebook 01's DCF: capitalize **2011 NACF at 5%**, receive the sale proceeds at the **end of 2010**, then discount the ten holding-year receipts at **7%**. The inherited sale convention uses NACF. This is asset value, with no purchase price, debt or equity IRR.

Define the calculation once so the reference case and sensitivity use the same timing and rates. The first receipt is discounted once and the last ten times. `collapse()` places the resulting value at the start of the holding span.


In [26]:
exit_rate = assumptions["timing"]["exit_capitalization_rate"]
discount_rate = assumptions["timing"]["annual_discount_rate"]


def value_nacf(flow):
    sale = Flow.from_events(
        dates=[date.fromisoformat(assumptions["timing"]["sale_date"])],
        magnitudes=[flow.movements[-1].number / exit_rate],
        units="AUD",
    )
    sale_annual = (
        Stream(flows={"Sale": sale})
        .resample(
            periods=holding_span.periods(frequency=Frequency.YEAR),
            method=ResamplingMethod.SUM,
            missing=MissingValueHandling.ZERO,
        )
        .sum()
    )
    receipts = Stream(
        flows={
            "NACF": flow.trim(
                start=holding_span.start_inclusive, end=holding_span.end_exclusive
            ),
            "Sale": sale_annual,
        }
    ).sum()
    return financial.calculate_pv(
        flow=receipts, rate=discount_rate, first_period=1
    ).collapse(on=holding_span.start_inclusive)


asset_value = value_nacf(property_nacf)
next_year_nacf = property_nacf.movements[-1].number
sale_value = next_year_nacf / exit_rate
pl.DataFrame(
    {
        "2011 NACF (AUD)": [next_year_nacf],
        "Sale, end 2010 (AUD)": [sale_value],
        "Asset value, start 2001 (AUD)": [asset_value.total().magnitude],
    }
).with_columns(pl.col(pl.Float64).round(2))


2011 NACF (AUD),"Sale, end 2010 (AUD)","Asset value, start 2001 (AUD)"
f64,f64,f64
"14,233,731.89","284,674,637.71","209,491,758.7"


## 8. Change one assumption and explain the result

Increase **residential rent by 10%**, holding all other inputs fixed. Predict which objects change before running the code. Copy the assumptions, recalculate income, then reuse the branch and valuation functions.

This is a controlled sensitivity: rent is specified and value is calculated. Asking which rent achieves a required value is a different problem, introduced in notebook 03.


In [27]:
variant_inputs = copy.deepcopy(use_inputs)
variant_inputs["residential"]["initial_rent_AUD_per_m2_per_year"] *= 1.10
variant_income = {e.id: floor_income(e, variant_inputs) for e in floors}
changed_ids = {
    uid
    for uid in income_by_floor
    if any(
        not math.isclose(a.number, b.number, abs_tol=1e-7)
        for a, b in zip(variant_income[uid].movements, income_by_floor[uid].movements)
    )
}
residential_ids = {e.id for e in floors if floor_use(e) == "residential"}
assert changed_ids == residential_ids
assert codec.dumps(model) == source_before


In [28]:
variant_nacf_by_branch = branch_cash_flows(variant_income)
variant_property_nacf = Stream(flows=variant_nacf_by_branch).sum()
reference_flows = {**nacf_by_branch, "Property": property_nacf}
variant_flows = {**variant_nacf_by_branch, "Property": variant_property_nacf}
comparison_rows = []
for label, ids in {**contributors_by_branch, "Property": set(income_by_floor)}.items():
    income_change = sum(
        variant_income[uid].movements[0].number
        - income_by_floor[uid].movements[0].number
        for uid in ids & set(income_by_floor)
    )
    reference_value = value_nacf(reference_flows[label]).total().magnitude
    variant_value = value_nacf(variant_flows[label]).total().magnitude
    comparison_rows.append(
        {
            "Component": label,
            "Affected floors": len(ids & changed_ids),
            "EGI change, 2001 (AUD)": income_change,
            "Reference value (AUD)": reference_value,
            "Variant value (AUD)": variant_value,
            "Change (AUD)": variant_value - reference_value,
        }
    )
comparison = pl.DataFrame(comparison_rows)
comparison.with_columns(pl.col(pl.Float64).round(2))


Component,Affected floors,"EGI change, 2001 (AUD)",Reference value (AUD),Variant value (AUD),Change (AUD)
str,i64,f64,f64,f64,f64
"""Building A""",4,"102,316.39","131,984,205.3","134,659,207.14","2,675,001.84"
"""Building B""",8,"507,199.3","98,893,428.65","112,153,856.19","13,260,427.54"
"""Shared plinth""",0,0,"-21,385,875.26","-21,385,875.26",0
"""Property""",12,"609,515.69","209,491,758.7","225,427,188.08","15,935,429.38"


Only **12 residential floor objects** change: four in Building A and eight in Building B. Higher future income also increases sale proceeds. The shared plinth is unchanged; its negative contribution is not a standalone market valuation of the basement.

We can now:

- Inspect one project through geometry, spatial structure and service relationships.
- Trace quantities and assumptions to flows, counting each contributor once.
- Explain a value change without changing the source Model.

**Next:** turn the completed property into a development problem. When is each part built, what must be available first, and what funding is needed? Notebook 03 introduces financial constraints and changes which quantities are fixed or unknown. Later lessons add market uncertainty, phasing and use-mix flexibility.

### Sources and conventions

This chapter combines the legacy [loading a design](https://daniel-fink.github.io/rangekeeper/load_design.html) and [driving valuation from design](https://daniel-fink.github.io/rangekeeper/drive_model_from_design.html) lessons. It retains their concept images and financial assumptions. Utility-only CAPEX is included here, correcting an omission in the legacy calculation.

The [manifest](data/mixed_use_property/manifest.json) pins the source captured on 12 October 2026. [Assumptions](data/mixed_use_property/assumptions.json) remain separate from [Model records](data/mixed_use_property/model.json). To rerun, use the complete walkthrough directory, including `helpers/`, data and resources. See the repository's `docs/guides/walkthroughs.md` for setup and validation.

The local [Speckle viewer](https://docs.speckle.systems/developers/viewer/introduction) reads the captured archive; the [hosted source](https://app.speckle.systems/projects/c0f66c35e3/models/119ad04487) reported an expired version at preparation. Display meshes support inspection; calculations use recorded Model measurements.